# PyTorch in Practice: Tensors, Initialization, Optimizers, Schedules, Regularization

*A one-hour, self-contained notebook for a guided session or self-study.*

You already know the basic training loop: forward pass, loss, `loss.backward()`, `optimizer.step()`. This notebook first sharpens two tensor skills you will use constantly, then covers the choices that decide whether the training loop trains **well**.

| § | Topic | After this section you can … | ≈ min |
| --- | --- | --- | --- |
| 0 | Setup | load the data and the small training helpers | 3 |
| **A** | **Tensor skills** | | |
| 1 | Tensor internals | explain storage, strides, views and contiguity; avoid the two classic view bugs | 9 |
| 2 | `einsum` | write tensor contractions directly in index notation | 6 |
| **B** | **Training well** | | |
| 3 | Weight initialization | explain why the initial weight scale matters and apply He/Xavier initialization | 6 |
| 4 | Optimizers | choose between SGD with momentum, Adam and AdamW, set their key parameters, and tune the learning rate | 11 |
| 5 | Learning-rate schedulers | pick a schedule for your situation and step it correctly | 10 |
| 6 | Weight decay | apply decoupled weight decay to the right parameters | 3 |
| 7 | Early stopping | implement early stopping that keeps the right weights, and use it on real data | 12 |

The minutes are a teaching plan, not a measurement. The tensor skills come first because they are short and independent, and because §7 relies on them.

**Requirements.** Python ≥ 3.10, PyTorch 2.x with torchvision, and matplotlib. The notebook uses only APIs that exist since PyTorch 2.0; it was executed end to end with **PyTorch 2.15** (the current release in November 2026) on a CPU. On Colab everything is preinstalled; elsewhere run `pip install torch torchvision matplotlib`.

**Hardware.** All training runs together take about a minute on a *single* CPU core, so any laptop is enough. A GPU is used automatically if present. The setup downloads Fashion-MNIST (about 30 MB) once.

**Conventions.** 🤔 *predict before you run*, ⚠️ *common pitfall*, 💡 *practical tip*. Numbers quoted in the text come from our test run; yours may differ slightly with hardware and library versions.

---
# 0. Setup
Run both cells. The first imports the libraries and picks a device.

In [ ]:
import copy, math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import lr_scheduler

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 0):
    print("⚠️  This notebook needs PyTorch 2.0 or newer; you have", torch.__version__)

torch.manual_seed(0)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

The second cell prepares **Fashion-MNIST**: 28×28 grayscale images of clothing in 10 classes. To keep every run fast on a CPU and make overfitting visible quickly, we train on a **10,000-image subset** of the official training set and validate on 5,000 *other* training images. The official 10,000 test images are used only once, at the very end of §7. The normalization statistics come from the training split only.

It also defines the two helpers every experiment uses: one training epoch (with optional per-batch scheduler, §5) and an evaluation in `eval` mode without autograd.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

# torchvision tries its own mirror first; the dataset's official GitHub repository is added as a fallback.
GITHUB_MIRROR = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
if GITHUB_MIRROR not in datasets.FashionMNIST.mirrors:
    datasets.FashionMNIST.mirrors = [*datasets.FashionMNIST.mirrors, GITHUB_MIRROR]
raw_train = datasets.FashionMNIST("./data", train=True, download=True)
raw_test = datasets.FashionMNIST("./data", train=False, download=True)

perm = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(0))
train_idx, val_idx = perm[:10_000], perm[10_000:15_000]       # disjoint subsets of the official training set

def as_vectors(images):
    return images.float().div(255).flatten(1)                  # uint8 (N, 28, 28) -> float (N, 784) in [0, 1]

x_train, y_train = as_vectors(raw_train.data[train_idx]), raw_train.targets[train_idx]
x_val, y_val = as_vectors(raw_train.data[val_idx]), raw_train.targets[val_idx]
x_test, y_test = as_vectors(raw_test.data), raw_test.targets
mean, std = x_train.mean(), x_train.std()                      # statistics of the TRAINING split only
x_train, x_val, x_test = [(x - mean) / std for x in (x_train, x_val, x_test)]

def make_train_loader(seed=0, batch_size=128):
    return DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=1000)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=1000)

def train_one_epoch(model, loader, optimizer, scheduler=None):
    # One pass over the training data. A scheduler passed here is stepped after EVERY batch (§5).
    model.train()
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader):
    # Mean cross-entropy and accuracy, in eval mode (dropout off, BatchNorm running statistics) and without autograd.
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

print(f"train {len(x_train):,} | val {len(x_val):,} | test {len(x_test):,} | classes: {raw_train.classes}")

---
# Part A · Tensor skills

# 1. Tensor internals: storage, strides and views

A tensor's numbers live in a flat, one-dimensional block of memory, the **storage**. The tensor itself is only a *description* of how to read that storage: a starting position (`storage_offset()`), a `shape`, and a `stride()`, the number of storage positions to jump for one step along each dimension. Element `[i, j]` lives at

$$\text{offset} + i\cdot\text{stride}[0] + j\cdot\text{stride}[1].$$

In [ ]:
def draw_storage_and_views():
    x = torch.arange(12).reshape(3, 4)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.set_xlim(-0.2, 13.2); ax.set_ylim(-6.8, 1.8); ax.set_aspect("equal"); ax.axis("off")
    def cell(cx, cy, value, color):
        ax.add_patch(plt.Rectangle((cx, cy), 1, 1, facecolor=color, edgecolor="gray"))
        ax.text(cx + 0.5, cy + 0.5, str(value), ha="center", va="center", fontsize=10)
    ax.text(0, 1.3, "storage (flat memory): positions 0 ... 11", fontsize=10)
    for k, v in enumerate(x.flatten().tolist()):
        cell(1 + k, 0, v, "#c6d7ee")
    ax.text(0.5, -0.5, "x = torch.arange(12).reshape(3, 4)\nshape (3, 4), stride (4, 1)", fontsize=9, va="top")
    for i in range(3):
        for j in range(4):
            cell(1 + j, -2.6 - i, x[i, j].item(), "white")
    ax.text(7.5, -0.5, "x.T\nshape (4, 3), stride (1, 4): same storage, no copy", fontsize=9, va="top")
    for i in range(4):
        for j in range(3):
            cell(8.5 + j, -2.6 - i, x.T[i, j].item(), "white")
    ax.text(6.6, -6.6, "storage position of element [i, j] = offset + i * stride[0] + j * stride[1]",
            ha="center", fontsize=10)
    plt.tight_layout(); plt.show()

draw_storage_and_views()

In [ ]:
x = torch.arange(12).reshape(3, 4)
print(x)
print("shape:", tuple(x.shape), "| stride:", x.stride(), "| offset:", x.storage_offset())

def address(t, idx):
    # Storage position of element t[idx], computed from the offset and the strides.
    return t.storage_offset() + sum(i * s for i, s in zip(idx, t.stride()))

print("x[2, 1] =", x[2, 1].item(), "lives at storage position", address(x, (2, 1)), "= 0 + 2*4 + 1*1")

For a freshly created (contiguous) tensor the last stride is 1 and each earlier stride is the product of all later sizes: shape `(2, 3, 4)` has strides `(12, 4, 1)`.

## 1.1 Views share memory
Many operations only change the description and return a **view** of the same storage. Views are free to create (no data is copied), and **writing through a view changes the original**.

| Returns a view (shares memory) | Returns a copy (new memory) |
| --- | --- |
| basic slicing `x[1]`, `x[:, 2]`, `x[::2]`; `view`; `transpose`, `permute`, `.T`, `.mT`; `expand`; `squeeze`, `unsqueeze`; `narrow`; `diagonal`; `detach` | indexing with an index tensor or a boolean mask, `x[[0, 2]]`, `x[x > 3]`; `clone`; `repeat`; `contiguous()` of a non-contiguous tensor; `reshape`/`flatten` *when a view is impossible* |

In [ ]:
x = torch.arange(12).reshape(3, 4)

def describe(name, t):
    shares = t.untyped_storage().data_ptr() == x.untyped_storage().data_ptr()
    print(f"{name:9s} shape={str(tuple(t.shape)):9s} stride={str(t.stride()):8s} "
          f"offset={t.storage_offset():2d}  shares memory with x: {shares}")

describe("x", x)
describe("x[1]", x[1])            # row 1: start at position 4, step 1
describe("x[:, 2]", x[:, 2])      # column 2: start at 2, step 4 (one whole row)
describe("x[::2]", x[::2])        # every other row: the row stride doubles to 8
describe("x.T", x.T)              # transpose: the two strides swap
describe("x[None]", x[None])      # a new size-1 dimension
describe("x[[0, 2]]", x[[0, 2]])  # indexing with a list/tensor: a copy

In [ ]:
x = torch.zeros(3, 4)
col = x[:, 2]           # a view
col += 7                # in-place write through the view ...
print(x)                # ... changes the original

safe = x[:, 2].clone()  # clone() = a real copy with its own storage
safe += 100
print("x[:, 2] after writing to the clone:", x[:, 2])

⚠️ **Bug 1: a "snapshot" that keeps changing.** Anything that hands you views, for example `model.state_dict()` (its tensors share storage with the parameters), gives you the *live* values. To keep a snapshot, copy it (`clone()`, `copy.deepcopy`). You will need this in §7.

⚠️ A small view keeps its **whole** storage alive: `first_rows = huge[:10]` keeps all of `huge` in memory. Use `.clone()` when you keep a small slice of a large tensor.

## 1.2 Contiguity, and why `view` sometimes fails
A tensor is **contiguous** if reading its elements in row-major order (last index fastest) walks through the storage as 0, 1, 2, …. A transpose is not contiguous:

In [ ]:
x = torch.arange(6).reshape(2, 3)
xt = x.T
print("x  =", x.tolist(), "stride", x.stride(), "contiguous:", x.is_contiguous())
print("xt =", xt.tolist(), "stride", xt.stride(), "contiguous:", xt.is_contiguous())
print("reading xt row by row visits storage positions:",
      [address(xt, (i, j)) for i in range(3) for j in range(2)])   # not 0, 1, 2, ...

`view` promises never to copy, so it fails when the new shape cannot be described with strides over the existing memory layout. The fixes:
- `.contiguous().view(...)`: first copy the data into a new, contiguous layout;
- `reshape(...)`: returns a view when possible and a copy otherwise. Use it unless you specifically need the guarantee that no copy is made.

In [ ]:
try:
    xt.view(6)
except RuntimeError as e:
    print("view failed:", str(e)[:100], "...")

flat1 = xt.contiguous().view(6)
flat2 = xt.reshape(6)
print("contiguous().view:", flat1.tolist(), "| reshape:", flat2.tolist())
print("reshape of the contiguous x is a view (same memory):", x.reshape(6).data_ptr() == x.data_ptr())
print("reshape of xt had to copy (new memory):           ", flat2.data_ptr() != x.data_ptr())

**Broadcasting is a stride of 0.** `expand` sets the stride of a stretched dimension to 0, so the same values are read again and again without any copy. `repeat` really copies. Because many elements of an expanded tensor share one memory location, PyTorch refuses in-place writes to it.

In [ ]:
b = torch.tensor([1.0, 2.0, 3.0])
big_view = b.expand(1000, 3)          # pretend there are 1000 rows
big_copy = b.repeat(1000, 1)          # really make 1000 rows
print("expand: shape", tuple(big_view.shape), "stride", big_view.stride(), "-> no new memory")
print("repeat: shape", tuple(big_copy.shape), "stride", big_copy.stride(),
      f"-> {big_copy.numel() * big_copy.element_size():,} bytes of new memory")

try:
    big_view += 1                     # 1000 'rows' would all write to the same 3 numbers
except RuntimeError as e:
    print("in-place write to an expanded tensor:", str(e)[:85], "...")

## 1.3 ⚠️ Bug 2: in-place writes through views and autograd
Some operations save their inputs or outputs for the backward pass; `exp`, for example, saves its output, because $\frac{d}{dx}e^x = e^x$. Every tensor has a *version counter* that increases with each in-place change, including changes made through a view. If a saved tensor has changed by the time `backward()` needs it, autograd stops with an error rather than silently computing a wrong gradient:

In [ ]:
a = torch.randn(4, requires_grad=True)
b = a.exp()               # exp saves its OUTPUT b for the backward pass
b[:2].zero_()             # in-place write through a view of b: b's version counter increases
try:
    b.sum().backward()
except RuntimeError as e:
    print("backward failed:", str(e)[:110], "...")

b = a.exp()
c = b.clone()             # fix: write into a copy, not into the saved tensor
c[:2].zero_()
c.sum().backward()
print("with clone(): a.grad =", a.grad)

**Takeaway.** A tensor = storage + offset + shape + strides. Views change only the description: they are free, and they alias the original. `view` needs a compatible layout, `reshape` copies when it must, and `clone` always gives you independent memory. Avoid in-place writes into tensors that autograd may still need.

---
# 2. `einsum`: index notation as code

Papers write formulas such as $y_{ni} = \sum_j x_{nj}W_{ij}$. `torch.einsum` lets you write them almost literally:
```python
y = torch.einsum("nj,ij->ni", x, W)
```

**Rules**
1. Each letter labels one dimension of one operand.
2. A letter that appears in several operands means those dimensions are aligned and **multiplied** element by element; their sizes must match.
3. A letter that does **not** appear after `->` is **summed** over.
4. The output dimensions appear in the order written after `->`.
5. `...` stands for "any number of leading dimensions", e.g. `"...ij,...jk->...ik"` is a matrix product batched over all leading dimensions.

`nn.Linear` computes $y = xW^\top$, so `"nj,ij->ni"` with `W = lin.weight` reproduces it:

In [ ]:
lin = nn.Linear(5, 3, bias=False)
x = torch.randn(4, 5)
y = torch.einsum("nj,ij->ni", x, lin.weight)          # y_ni = sum_j x_nj W_ij
print(tuple(y.shape), "| matches nn.Linear:", torch.allclose(y, lin(x), atol=1e-6))

| Operation | `einsum` | Plain PyTorch |
| --- | --- | --- |
| transpose | `"ij->ji"` | `A.T` |
| sum of all / row sums | `"ij->"` / `"ij->i"` | `A.sum()` / `A.sum(dim=1)` |
| trace / diagonal | `"ii->"` / `"ii->i"` | `A.trace()` / `A.diagonal()` |
| element-wise product | `"ij,ij->ij"` | `A * B` |
| dot / outer product | `"i,i->"` / `"i,j->ij"` | `u @ v` / `torch.outer(u, v)` |
| matrix–vector / matrix–matrix | `"ij,j->i"` / `"ij,jk->ik"` | `A @ u` / `A @ B` |
| batched matrix product | `"bij,bjk->bik"` | `A @ B` (3-D) |
| row-wise dot products | `"nd,nd->n"` | `(X * Y).sum(dim=1)` |
| per-example bilinear form $x_n^\top M y_n$ | `"ni,ij,nj->n"` | `((x @ M) * y).sum(dim=1)` |

Every row, checked against the plain version:

In [ ]:
A, B = torch.randn(3, 3), torch.randn(3, 3)
u, v = torch.randn(3), torch.randn(3)
X, Y = torch.randn(5, 4), torch.randn(5, 4)
M = torch.randn(4, 4)
Ab, Bb = torch.randn(2, 3, 4), torch.randn(2, 4, 5)

checks = {
    "transpose":      (torch.einsum("ij->ji", A), A.T),
    "sum":            (torch.einsum("ij->", A), A.sum()),
    "row sums":       (torch.einsum("ij->i", A), A.sum(dim=1)),
    "trace":          (torch.einsum("ii->", A), A.trace()),
    "diagonal":       (torch.einsum("ii->i", A), A.diagonal()),
    "elementwise":    (torch.einsum("ij,ij->ij", A, B), A * B),
    "dot":            (torch.einsum("i,i->", u, v), u @ v),
    "outer":          (torch.einsum("i,j->ij", u, v), torch.outer(u, v)),
    "matrix-vector":  (torch.einsum("ij,j->i", A, u), A @ u),
    "matrix-matrix":  (torch.einsum("ij,jk->ik", A, B), A @ B),
    "batched matmul": (torch.einsum("bij,bjk->bik", Ab, Bb), Ab @ Bb),
    "row-wise dots":  (torch.einsum("nd,nd->n", X, Y), (X * Y).sum(dim=1)),
    "bilinear":       (torch.einsum("ni,ij,nj->n", X, M, Y), ((X @ M) * Y).sum(dim=1)),
}
for name, (a, b) in checks.items():
    print(f"{name:15s} shape {str(tuple(a.shape)):10s} agrees: {torch.allclose(a, b, atol=1e-5)}")

## 2.1 Two practical examples
**(a) All pairwise squared distances** between $N$ points $X$ and $M$ points $Y$ (used in k-nearest neighbors and kernel methods). Expanding $\lVert x_n - y_m\rVert^2 = \lVert x_n\rVert^2 + \lVert y_m\rVert^2 - 2\,x_n\!\cdot y_m$ turns the expensive part into one contraction, $\sum_d X_{nd}Y_{md}$.

**(b) An ensemble of $K$ linear models applied at once.** With weights $W_{koi}$ (model $k$, output $o$, input $i$) and inputs $x_{ni}$, all $K$ models' outputs are $y_{kno} = \sum_i W_{koi}\,x_{ni}$, with no Python loop.

In [ ]:
X, Y = torch.randn(6, 3), torch.randn(4, 3)
D2 = (X ** 2).sum(1)[:, None] + (Y ** 2).sum(1)[None, :] - 2 * torch.einsum("nd,md->nm", X, Y)
print("(a) pairwise squared distances:", tuple(D2.shape), "| agrees with torch.cdist:",
      torch.allclose(D2, torch.cdist(X, Y) ** 2, atol=1e-4))

K, n_in, n_out, N = 5, 8, 3, 10
W = torch.randn(K, n_out, n_in)
x = torch.randn(N, n_in)
y_all = torch.einsum("koi,ni->kno", W, x)                         # all K models in one call
y_loop = torch.stack([x @ W[k].T for k in range(K)])              # the same with a loop
print("(b) ensemble outputs:", tuple(y_all.shape), "| agrees with the loop:", torch.allclose(y_all, y_loop, atol=1e-5))

batched = torch.einsum("...ij,...jk->...ik", torch.randn(2, 3, 4, 5), torch.randn(2, 3, 5, 6))
print("(c) '...' batches over any leading dimensions:", tuple(batched.shape))

You will meet `einsum` again in attention layers later in the course, where strings such as `"bhqd,bhkd->bhqk"` compare every query with every key.

**Performance notes**
- For two operands, PyTorch turns an `einsum` into a (batched) matrix multiplication, so it is as fast as `@`.
- For three or more operands, the order of the contractions can change the cost enormously. PyTorch optimizes that order when the `opt_einsum` package is installed (check with `torch.backends.opt_einsum.is_available()`); otherwise it contracts from left to right.
- For pure rearrangements (no products), the third-party **einops** library is popular, e.g. `rearrange(x, "b c h w -> b (h w) c")`.

**Takeaway.** Label every dimension with a letter; shared letters are multiplied, letters missing after `->` are summed.

---
# Part B · Training well

# 3. Weight initialization (brief overview)

## 3.1 Why the initial scale matters
A layer computes $z = Wa + b$. In a deep network the scale of the signal is multiplied by roughly the same factor at **every** layer. A factor below 1 makes the input signal **vanish** and, with it, the gradients that reach the early layers; a factor above 1 makes them **explode**. Initialization chooses the weight variance so that this factor is close to 1 *at the start of training*.

For $n_\text{in}$ inputs and independent, zero-mean weights (ignoring the bias),
$$\operatorname{Var}(z_i) = n_\text{in}\,\operatorname{Var}(w)\,\mathbb{E}[a^2].$$
Keeping this constant from layer to layer gives the two classic rules:

- **He (Kaiming) initialization** for ReLU-type activations: ReLU sets half of a symmetric input to zero, which halves $\mathbb{E}[a^2]$. Compensate with $\operatorname{Var}(w) = 2/n_\text{in}$.
- **Xavier (Glorot) initialization** for tanh, sigmoid or no activation: $\operatorname{Var}(w) = 2/(n_\text{in}+n_\text{out})$, a compromise between keeping the forward signal and the backward gradient at scale. It is multiplied by a *gain* that depends on the activation (5/3 for tanh).

These are approximate arguments about the starting point, not guarantees for the whole of training.

| Initialization | Weights | PyTorch call | Typical use |
| --- | --- | --- | --- |
| PyTorch default (`nn.Linear`, `nn.Conv*d`) | $\mathcal U(-1/\sqrt{n_\text{in}},\,1/\sqrt{n_\text{in}})$; biases from the same range | automatic | shallow nets; nets with normalization layers; output layers |
| He / Kaiming | $\mathcal N(0,\,2/n_\text{in})$ | `nn.init.kaiming_normal_(w, nonlinearity="relu")` | ReLU; also commonly used for GELU/SiLU |
| Xavier / Glorot | uniform with variance $2/(n_\text{in}+n_\text{out})$ | `nn.init.xavier_uniform_(w, gain=nn.init.calculate_gain("tanh"))` | tanh, sigmoid, linear |
| Small normal | $\mathcal N(0,\,0.02^2)$ | `nn.init.normal_(w, std=0.02)` | GPT-2-style transformers |
| Zeros | 0 | `nn.init.zeros_(b)` | biases |

For a convolution, $n_\text{in} = C_\text{in}\cdot k_h \cdot k_w$; PyTorch computes this "fan-in" for you.

⚠️ Vision-transformer code (e.g. timm) uses `nn.init.trunc_normal_(w, std=0.02)`. Its cut-offs `a=-2.0, b=2.0` are **absolute values**, not multiples of `std`, so with `std=0.02` nothing is actually truncated. For a ±2σ cut-off, pass `a=-2 * std, b=2 * std`.

💡 **Normalization layers** (BatchNorm, LayerNorm) and **residual connections** make modern networks much less sensitive to initialization, so the default is often fine. A custom initialization matters most for deep stacks without normalization, and when you reproduce a published recipe.

## 3.2 Seeing it: a 20-layer ReLU network
We push a batch of 1,024 random inputs through 20 `Linear + ReLU` layers and a final `Linear` output layer. We record two things:
- **left:** after each ReLU, how much the activations *differ between inputs* (the standard deviation across the batch, averaged over the units). This is the part of the signal that carries information about the input;
- **right:** the norm of each layer's weight gradient for a cross-entropy loss with random labels.

Only the hidden-layer weights differ between the three versions; the biases and the output layer keep their defaults.

🤔 **Predict:** which initialization keeps both the input signal and the gradients alive through 20 layers?

In [ ]:
def deep_relu_mlp(depth=20, width=512):
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(width, width), nn.ReLU()]
    return nn.Sequential(*layers, nn.Linear(width, 10))

def activation_stds(model, x):
    stds = []
    with torch.no_grad():
        for layer in model:
            x = layer(x)
            if isinstance(layer, nn.ReLU):
                stds.append(x.std(dim=0).mean().item())      # spread across inputs, averaged over units
    return stds

def weight_grad_norms(model, x, y):
    model.zero_grad()
    F.cross_entropy(model(x), y).backward()
    return [m.weight.grad.norm().item() for m in model.modules() if isinstance(m, nn.Linear)]

weight_inits = {
    "PyTorch default": lambda w: None,                            # keep the default weights
    "normal, std = 0.01": lambda w: nn.init.normal_(w, std=0.01),
    "He (kaiming_normal_)": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
}
x, y = torch.randn(1024, 512), torch.randint(0, 10, (1024,))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
for name, init_w in weight_inits.items():
    torch.manual_seed(0)
    net = deep_relu_mlp()
    for m in list(net.modules())[1:-1]:                           # hidden layers only; the output layer keeps its default
        if isinstance(m, nn.Linear):
            init_w(m.weight)
    a1.semilogy(range(1, 21), activation_stds(net, x), "o-", ms=3, label=name)
    a2.semilogy(range(1, 22), weight_grad_norms(net, x, y), "o-", ms=3, label=name)
a1.set(xlabel="ReLU layer", ylabel="std across inputs (log)", title="Forward: input-dependent signal")
a2.set(xlabel="Linear layer (input → output)", ylabel="weight-gradient norm (log)", title="Backward: gradient per layer")
a1.legend(fontsize=8); plt.tight_layout(); plt.show()

- **std = 0.01 and PyTorch default** (whose $\operatorname{Var}(w) = 1/(3n_\text{in})$ is too small for a deep ReLU stack): each layer shrinks the input-dependent signal by a roughly constant factor, so after 20 layers every input produces practically the same activations. (The activations themselves do not become zero: the randomly initialized biases keep them at a small, input-independent level.) The gradients reaching the first layers are about 8 (default) and 16 (std = 0.01) orders of magnitude smaller than at the output, so the early layers cannot learn.
- **He:** the input-dependent signal stays within the same order of magnitude through all 20 layers, and the gradients reach the first layer. It still decreases slowly: very deep plain ReLU stacks make different inputs gradually more alike, one of the reasons modern networks add residual connections and normalization.

## 3.3 How to apply an initialization
Write a function for one module and let `model.apply(fn)` call it on every submodule. The `nn.init.*_` functions work **in place** (trailing underscore) and are not recorded by autograd.

💡 Leave the **output layer** at its default (smaller) scale. He initialization there makes the initial logits large, so the model starts out confidently wrong. A useful sanity check: with 10 balanced classes, the initial cross-entropy should be close to $\ln 10 \approx 2.30$.

The MLP below (784 → 512 → 512 → 10) is used in all experiments of §4–7.

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")   # He initialization
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def make_mlp(hidden=512):
    model = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(),
                          nn.Linear(hidden, hidden), nn.ReLU(),
                          nn.Linear(hidden, 10))
    model[:-1].apply(init_weights)            # He for the hidden layers; the output layer keeps the default
    return model.to(device)

torch.manual_seed(0)
model = make_mlp()
print(f"std of first-layer weights: {model[0].weight.std():.4f}  (He target sqrt(2/784) = {math.sqrt(2 / 784):.4f})")
print(f"initial validation loss, He hidden + default output: {evaluate(model, val_loader)[0]:.2f}")
torch.manual_seed(0)
model_he_everywhere = make_mlp().apply(init_weights)
print(f"initial validation loss, He on every layer:          {evaluate(model_he_everywhere, val_loader)[0]:.2f}"
      f"   (ln 10 = {math.log(10):.2f})")

**Takeaway.** Use He initialization for ReLU-type hidden layers and Xavier for tanh/sigmoid, keep the output layer small, and check that the initial loss is near $\ln(\text{number of classes})$. With normalization layers or shallow networks, the PyTorch default is usually fine.

---
# 4. Optimizers

## 4.1 What an optimizer is
An optimizer updates the parameters from their gradients. It stores two things:
- **`param_groups`**: a list of dicts, each holding some parameters and their hyper-parameters (`lr`, `weight_decay`, …). Different groups can have different settings (used in §6).
- **`state`**: per-parameter memory, such as momentum buffers. It is created at the first `step()`, and it must be saved in a checkpoint if training is to be resumed exactly.

Below, $g$ is a parameter's gradient and $\eta$ the learning rate (`lr`).

## 4.2 SGD with momentum
PyTorch's form keeps a running **sum** of past gradients:
$$v \leftarrow \mu v + g, \qquad p \leftarrow p - \eta v.$$
Where the gradient keeps its sign, the terms add up (faster progress); where it oscillates, they cancel (damping). For a constant gradient $v \to g/(1-\mu)$, so with $\mu = 0.9$ the step is about **10× larger** than plain SGD with the same `lr`. With `nesterov=True` the update looks one step ahead: $p \leftarrow p - \eta(g + \mu v)$.

`torch.optim.SGD(params, lr, momentum=0.9, nesterov=True, weight_decay=...)`
- `lr`: **must be tuned** (§4.8). A classic image-classification recipe uses `lr=0.1` at batch size 256, together with a schedule (§5).
- `momentum`: 0.9 is the standard value. The default is 0, which gives plain SGD.

## 4.3 Adam
Adam keeps exponential moving averages of the gradient and of its square, corrects their bias toward zero in the first steps, and divides one by the square root of the other:
$$m \leftarrow \beta_1 m + (1-\beta_1)g,\qquad v \leftarrow \beta_2 v + (1-\beta_2)g^2,\qquad \hat m = \frac{m}{1-\beta_1^t},\quad \hat v = \frac{v}{1-\beta_2^t},$$
$$p \leftarrow p - \eta\,\frac{\hat m}{\sqrt{\hat v}+\varepsilon}.$$
When a parameter's gradient is consistent, it moves by about $\eta$ per step, whatever the scale of its gradient; when the gradient is noisy, $|\hat m| < \sqrt{\hat v}$ and the step is smaller. Because each parameter is scaled separately, Adam copes well with parameters whose gradients differ by orders of magnitude, which is common in neural networks. Its default `lr` works reasonably often, which makes it easy to use.

`torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. ⚠️ Adam's `weight_decay` is *L2 regularization*, which does not work as intended with Adam (§6). Use **AdamW** when you want weight decay.

## 4.4 AdamW
AdamW is Adam with **decoupled weight decay**: after the Adam step, every weight is also shrunk, $p \leftarrow p - \eta\lambda p$ (§6). It is a robust first choice almost everywhere and the standard optimizer for transformers. In current PyTorch, `AdamW` is implemented as `Adam(..., decoupled_weight_decay=True)`.

`torch.optim.AdamW(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)`. Note that the default `weight_decay` is 0.01 here, but 0 in `Adam`.

## 4.5 Which one, with which settings?

| Optimizer | Typical starting values | When |
| --- | --- | --- |
| **AdamW** | `lr=1e-3` for small MLPs/CNNs, `3e-4` or lower for transformers; `weight_decay=0.01–0.1`; large transformers often use `betas=(0.9, 0.95)` | the default first choice |
| **SGD + momentum** | `momentum=0.9`, `nesterov=True`, `lr≈0.1` at batch 256 (scaled roughly in proportion to the batch size), `weight_decay=1e-4–5e-4`, always with a schedule | CNNs for vision with established recipes, where it can generalize as well as or better than AdamW |
| **Adam** | `lr=1e-3`, default betas | when you want no weight decay at all (then identical to AdamW with `weight_decay=0`) |

💡 **Memory:** SGD with momentum stores one extra tensor per parameter, Adam/AdamW two (`exp_avg`, `exp_avg_sq`). On a GPU, `fused=True` makes the optimizer step faster for `SGD`, `Adam` and `AdamW`.

**Outlook.** `torch.optim` contains other optimizers too (RMSprop, NAdam, RAdam, Adagrad, LBFGS, …). One is worth knowing about: **`torch.optim.Muon`** (in PyTorch since 2.9) orthogonalizes the momentum update of 2-D weight matrices and is used *together with* AdamW for all other parameters. It has been adopted for some large language models, e.g. Moonshot AI's Kimi K2 (with an added clipping technique). AdamW remains the standard default.

## 4.6 Look inside

In [ ]:
lin = nn.Linear(3, 2)
for opt in [torch.optim.SGD(lin.parameters(), lr=0.1, momentum=0.9), torch.optim.AdamW(lin.parameters(), lr=1e-3)]:
    lin.zero_grad(); lin(torch.randn(8, 3)).pow(2).mean().backward(); opt.step()
    print(f"{type(opt).__name__:6s} state per weight tensor after one step: {list(opt.state[lin.weight].keys())}")

## 4.7 A fair comparison on a toy problem
The loss $f(p) = \tfrac12\,p^\top H p$ with curvatures 1 and 50 is a long, narrow valley: steep in one direction, shallow in the other. Very different curvatures in different directions are typical of neural-network losses. We use the valley twice: once with its axes **aligned** with the coordinate axes, and once **rotated by 45°**.

A comparison of optimizers is only fair if each one gets its own best learning rate. So for each optimizer and each valley we try 12 learning rates (0.001 to 2.048, each twice the previous), run 60 steps, and keep the learning rate with the lowest **mean loss over the last 20 steps** (more robust than the single last value, because some optimizers oscillate).

🤔 **Predict:** which optimizers behave identically on the aligned and the rotated valley?

In [ ]:
def valley(angle_deg):
    a = math.radians(angle_deg)
    R = torch.tensor([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
    H = R @ torch.diag(torch.tensor([1.0, 50.0])) @ R.T            # curvatures 1 and 50, axes rotated by angle
    start = R @ torch.tensor([-4.0, 1.0])                            # the same starting point, rotated along
    return H, start

def run(make_opt, H, start, steps=60):
    p = start.clone().requires_grad_(True)
    opt = make_opt([p])
    path, losses = [p.detach().clone()], []
    for _ in range(steps):
        loss = 0.5 * p @ H @ p
        losses.append(loss.item())
        opt.zero_grad(); loss.backward(); opt.step()
        path.append(p.detach().clone())
    return torch.stack(path), losses

optimizers = {
    "SGD": lambda lr: (lambda ps: torch.optim.SGD(ps, lr=lr)),
    "SGD + momentum 0.9": lambda lr: (lambda ps: torch.optim.SGD(ps, lr=lr, momentum=0.9)),
    "Adam": lambda lr: (lambda ps: torch.optim.Adam(ps, lr=lr)),
}
lr_grid = [0.001 * 2 ** k for k in range(12)]

fig, axes = plt.subplots(2, 3, figsize=(15, 6.4))
for row, angle in enumerate((0, 45)):
    H, start = valley(angle)
    gx, gy = torch.meshgrid(torch.linspace(-4.5, 4.5, 200), torch.linspace(-4.5, 4.5, 200), indexing="xy")
    grid = torch.stack([gx, gy], dim=-1)
    f_grid = 0.5 * torch.einsum("...i,ij,...j->...", grid, H, grid)   # einsum from §2
    for col, (name, make) in enumerate(optimizers.items()):
        results = []
        for lr in lr_grid:
            path, losses = run(make(lr), H, start)
            score = sum(losses[-20:]) / 20
            if math.isfinite(score):                                 # skip learning rates that diverged
                results.append((score, lr, path))
        score, lr, path = min(results, key=lambda r: r[0])
        ax = axes[row, col]
        ax.contour(gx, gy, f_grid, levels=[0.1, 0.5, 2, 5, 10, 20, 40, 80], cmap="Greys", linewidths=0.6)
        ax.plot(path[:, 0], path[:, 1], "o-", ms=2, lw=0.8)
        ax.plot(0, 0, "r*", ms=12); ax.set_aspect("equal"); ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)
        ax.set_title(f"{'aligned' if angle == 0 else 'rotated 45°'} | {name}\nbest lr {lr:g}, mean late loss {score:.1e}",
                     fontsize=9)
        print(f"{'aligned' if angle == 0 else 'rotated':8s} {name:19s} best lr {lr:<6g} mean loss (last 20 steps) {score:.1e}")
plt.tight_layout(); plt.show()

What the numbers show (each optimizer with its own best learning rate):
- **SGD and SGD with momentum give exactly the same result on both valleys.** Their update $p \leftarrow p - \eta v$ does not depend on how the coordinate axes are oriented (they are *rotation-invariant*). Momentum reaches a clearly lower loss than plain SGD, whose learning rate is capped by the steep direction (it diverges above $2/50 = 0.04$).
- **Adam's behavior depends on the orientation.** Its per-coordinate scaling acts along the parameter axes, so it helps most when the steep and shallow directions line up with them. Rotated, Adam needs a different best learning rate, follows a different path and ends at a higher loss than on the aligned valley; on this toy problem it is still competitive.
- **The learning rate matters at least as much as the choice of optimizer.** With a badly chosen learning rate, every optimizer here either crawls or diverges. Comparisons that give all optimizers the same learning rate are not fair.

A 2-D quadratic is a cartoon. Adam's practical advantage in neural networks is usually explained by the very different gradient scales of different parameters and layers, which a per-parameter scaling handles well.

## 4.8 Choosing the learning rate: a short sweep
The learning rate is the most important hyper-parameter, so tune it, at least roughly:
1. Try values on a **logarithmic grid**, about 3× apart (here 1e-4 … 1e-2).
2. Train each for a **short** time (here 2 epochs) and compare the **validation** loss, never the test loss.
3. Take the best value and confirm it with a full run. If the best value is at the edge of the grid, extend the grid.

The best value after a short run is not guaranteed to be the best for a long run; the sweep is for finding the right **order of magnitude**.

In [ ]:
sweep_lrs = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
sweep_val = []
for lr in sweep_lrs:
    torch.manual_seed(0)
    model = make_mlp()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loader = make_train_loader(seed=0)
    for _ in range(2):
        train_one_epoch(model, loader, optimizer)
    val_loss, val_acc = evaluate(model, val_loader)
    sweep_val.append(val_loss)
    print(f"lr {lr:.0e}: validation loss {val_loss:.4f}, accuracy {val_acc:.4f}")

plt.figure(figsize=(5.5, 3.2))
plt.semilogx(sweep_lrs, sweep_val, "o-")
plt.xlabel("learning rate (log scale)"); plt.ylabel("validation loss after 2 epochs")
plt.title("AdamW learning-rate sweep"); plt.tight_layout(); plt.show()
print("best learning rate in the sweep:", sweep_lrs[min(range(len(sweep_lrs)), key=sweep_val.__getitem__)])

The curve is U-shaped: too small and training is slow, too large and it becomes unstable. For this MLP and AdamW the minimum was at `1e-3` in our run, so that value is used below.

**Takeaway.** Start with AdamW (`lr=1e-3`, `weight_decay=0.01`) and tune the learning rate with a short log-scale sweep on the validation set. Use SGD with momentum 0.9 and a schedule when you follow an established vision recipe.

---
# 5. Learning-rate schedulers

## 5.1 Why and how
Early in training, large steps make fast progress; late in training, small steps let the weights settle into a good minimum. Many recipes also **warm up**: they start with a small learning rate for the first steps, because updates computed from random weights (and, for Adam, from noisy early estimates of $v$) can be large and destabilizing.

A scheduler wraps an optimizer and writes a new `lr` into `optimizer.param_groups` each time you call `scheduler.step()`.

**Rules that prevent most scheduler bugs**
1. Call `scheduler.step()` **after** `optimizer.step()`.
2. Decide whether you step **per batch or per epoch**, and size the schedule in the **same unit**: `CosineAnnealingLR(opt, T_max=epochs)` stepped per epoch, or `T_max=epochs * len(train_loader)` stepped per batch.
3. `ReduceLROnPlateau` is the exception: step it once per epoch **with the monitored value**, `scheduler.step(val_loss)`.
4. Log or plot the learning rate (`scheduler.get_last_lr()` or `optimizer.param_groups[0]["lr"]`). Most scheduler bugs are obvious in such a plot.

## 5.2 The most important schedulers and when to use them

| Scheduler | What it does | Use it when |
| --- | --- | --- |
| `StepLR` / `MultiStepLR` | multiply `lr` by `gamma` (e.g. 0.1) every `step_size` epochs / at given `milestones` | reproducing classic recipes (e.g. ResNet with SGD) |
| `CosineAnnealingLR` | smooth decay from `lr` to `eta_min` along half a cosine over `T_max` steps | you know the training length; a strong, simple default |
| linear warm-up + cosine: `SequentialLR([LinearLR, CosineAnnealingLR])` | ramp up over the first steps, then cosine decay | AdamW, transformers, large batches: the standard modern recipe (fine-tuning often uses warm-up + *linear* decay) |
| warm-up–stable–decay (WSD): `SequentialLR([LinearLR, ConstantLR, LinearLR])` | warm up, hold constant, then decay quickly over roughly the last 10–20 % of training | large-model pre-training (popular since 2024, e.g. MiniCPM): the constant phase can be extended, and checkpoints from it can be decayed later |
| `OneCycleLR` | warm up to `max_lr`, then anneal far below the start; **stepped per batch** | short runs where you want fast convergence; fixed total number of steps |
| `ReduceLROnPlateau` | multiply `lr` by `factor` when the validation metric has not improved for `patience` epochs | unknown training length; pairs naturally with early stopping (§7) |

The plot shows the first five over 100 scheduler steps with a base `lr` of 1:

In [ ]:
def lr_trace(make_scheduler, steps=100, base_lr=1.0):
    # The learning rate used at each step, for a dummy optimizer.
    opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=base_lr)
    sched = make_scheduler(opt)
    lrs = []
    for _ in range(steps):
        lrs.append(opt.param_groups[0]["lr"])   # lr used for this step
        opt.step()                              # (no gradients here, so nothing changes)
        sched.step()                            # AFTER optimizer.step()
    return lrs

S = lr_scheduler
schedules = {
    "MultiStepLR(milestones=[50, 75], gamma=0.1)":
        lambda o: S.MultiStepLR(o, milestones=[50, 75], gamma=0.1),
    "CosineAnnealingLR(T_max=100)":
        lambda o: S.CosineAnnealingLR(o, T_max=100),
    "warm-up (10) + cosine (90)":
        lambda o: S.SequentialLR(o, schedulers=[S.LinearLR(o, start_factor=0.01, total_iters=10),
                                                S.CosineAnnealingLR(o, T_max=90)], milestones=[10]),
    "WSD: warm-up 10, stable 70, decay 20":
        lambda o: S.SequentialLR(o, schedulers=[S.LinearLR(o, start_factor=0.01, total_iters=10),
                                                S.ConstantLR(o, factor=1.0, total_iters=70),
                                                S.LinearLR(o, start_factor=1.0, end_factor=0.0, total_iters=20)],
                                 milestones=[10, 80]),
    "OneCycleLR(max_lr=1, total_steps=100)":
        lambda o: S.OneCycleLR(o, max_lr=1.0, total_steps=100),
}
fig, axes = plt.subplots(1, 5, figsize=(18, 2.9), sharey=True)
for ax, (name, make) in zip(axes, schedules.items()):
    ax.plot(lr_trace(make)); ax.set_title(name, fontsize=8); ax.set_xlabel("scheduler step")
axes[0].set_ylabel("learning rate"); plt.tight_layout(); plt.show()

Details worth knowing:
- `LinearLR` needs `start_factor > 0`; the warm-up starts at `start_factor × lr`.
- `OneCycleLR` **overrides** the optimizer's `lr`: it starts at `max_lr / div_factor` (default 25) and ends at that value divided by `final_div_factor` (default 10⁴). Stepping it more than `total_steps` times raises an error.
- ⚠️ `OneCycleLR` also **cycles momentum** (or Adam's β₁) between 0.85 and 0.95 by default (`cycle_momentum=True`). On `SGD(momentum=0)` this silently **turns momentum on**:

In [ ]:
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)            # plain SGD, momentum = 0
_ = lr_scheduler.OneCycleLR(opt, max_lr=1.0, total_steps=100)
print("momentum after creating OneCycleLR:", opt.param_groups[0]["momentum"])
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
_ = lr_scheduler.OneCycleLR(opt, max_lr=1.0, total_steps=100, cycle_momentum=False)
print("with cycle_momentum=False:         ", opt.param_groups[0]["momentum"])

## 5.3 `ReduceLROnPlateau` reacts to the validation loss
It does not follow a fixed curve. Here it is fed a made-up sequence of validation losses that stalls twice:

In [ ]:
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
plateau = lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
val_losses = [1.00, 0.80, 0.70, 0.71, 0.70, 0.72, 0.65, 0.60, 0.61, 0.62, 0.61, 0.60]
for epoch, val_loss in enumerate(val_losses, start=1):
    opt.step()                    # (training for one epoch would happen here)
    plateau.step(val_loss)        # once per epoch, WITH the monitored value
    print(f"epoch {epoch:2d} | val loss {val_loss:.2f} | lr for the next epoch: {opt.param_groups[0]['lr']:.4f}")

After `patience=2` epochs without improvement the learning rate is halved, and the count starts again. By default "improvement" means at least 0.01 % better than the best value (`threshold=1e-4`, `threshold_mode="rel"`).

## 5.4 ⚠️ The most common bug: mixing units
🤔 **Predict:** a cosine schedule sized for 10 *epochs* (`T_max=10`) but stepped after every *batch*, with 50 batches per epoch.

In [ ]:
wrong = lr_trace(lambda o: S.CosineAnnealingLR(o, T_max=10), steps=10 * 50)
right = lr_trace(lambda o: S.CosineAnnealingLR(o, T_max=10 * 50), steps=10 * 50)
plt.figure(figsize=(8, 2.8))
plt.plot(wrong, label="T_max=10 (epochs), stepped every batch: BUG")
plt.plot(right, label="T_max=500 (batches), stepped every batch: correct")
plt.xlabel("batch"); plt.ylabel("lr"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

The buggy schedule oscillates 25 times instead of decaying once. No error and no warning is raised; only a plot of the learning rate reveals it.

## 5.5 Does the schedule matter? A real run
We train the same MLP for 10 epochs with AdamW (`lr=1e-3`), once with a **constant** learning rate and once with **one epoch of linear warm-up followed by cosine decay**, stepped after every **batch** (so the schedule is sized in batches: `10 * len(loader)`). To see whether a difference is real or luck, each version runs with **two seeds**.

In [ ]:
EPOCHS = 10
schedule_runs = {}
for seed in (0, 1):
    for name in ("constant", "warm-up + cosine"):
        torch.manual_seed(seed)
        model = make_mlp()
        loader = make_train_loader(seed)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
        scheduler = None
        if name == "warm-up + cosine":
            warmup, total = len(loader), EPOCHS * len(loader)              # sized in BATCHES
            scheduler = lr_scheduler.SequentialLR(optimizer, milestones=[warmup], schedulers=[
                lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup),
                lr_scheduler.CosineAnnealingLR(optimizer, T_max=total - warmup)])
        val_curve, lr_curve = [], []
        for epoch in range(EPOCHS):
            lr_curve.append(optimizer.param_groups[0]["lr"])
            train_one_epoch(model, loader, optimizer, scheduler)              # scheduler stepped per batch
            val_curve.append(evaluate(model, val_loader))
        schedule_runs[(name, seed)] = (val_curve, lr_curve)
        print(f"seed {seed} | {name:17s} | final validation loss {val_curve[-1][0]:.4f}, accuracy {val_curve[-1][1]:.4f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.4))
for (name, seed), (val_curve, lr_curve) in schedule_runs.items():
    style = dict(color="tab:blue" if name == "constant" else "tab:orange", ls="-" if seed == 0 else "--")
    a1.plot(range(1, EPOCHS + 1), [v[0] for v in val_curve], marker="o", ms=3, label=f"{name}, seed {seed}", **style)
    a2.plot(range(1, EPOCHS + 1), lr_curve, marker="o", ms=3, **style)
a1.set(xlabel="epoch", ylabel="validation loss", title="Validation loss"); a1.legend(fontsize=8)
a2.set(xlabel="epoch", ylabel="lr at the start of the epoch", title="Learning rate"); plt.tight_layout(); plt.show()

In our run, warm-up + cosine ended with a clearly lower validation loss than the constant learning rate for **both** seeds (about 0.36 versus about 0.42), and with slightly higher accuracy. With a constant learning rate the validation loss stops improving after about three epochs and jumps around (one seed shows a large spike); the decaying learning rate lets the weights settle and the validation loss keeps falling smoothly. Two seeds are the bare minimum for such a claim; in a report, use more and show the spread.

**How to choose, in one line each**
- Fixed training budget → **cosine** (with a linear **warm-up** for AdamW, transformers or large batches).
- Large-scale pre-training where the budget may change → **WSD**.
- Short run, want fast convergence → **OneCycleLR**, stepped per batch.
- Unknown number of epochs, or you use early stopping → **ReduceLROnPlateau**.
- Reproducing a paper → whatever the paper used.

---
# 6. Weight decay (very brief)

Weight decay shrinks every weight a little toward zero at each step, $p \leftarrow p - \eta\lambda p$. This discourages large weights and often improves generalization (it does not guarantee it: the validation set decides).

- **AdamW:** `torch.optim.AdamW(..., weight_decay=λ)` applies exactly this decoupled shrink, using the *current* learning rate. Typical λ: 0.01–0.1.
- **SGD:** `torch.optim.SGD(..., weight_decay=λ)` adds $\lambda p$ to the gradient. For plain SGD this is the same thing. Typical λ: 1e-4–5e-4.
- ⚠️ **Not `Adam(weight_decay=λ)`.** Adam adds $\lambda p$ to the gradient and then divides by $\sqrt{\hat v}$, so weights with large gradients are barely regularized. Use `AdamW`.
- The shrink per step is $\eta\lambda$, so the total effect grows with the learning rate and the number of steps. Retune λ when you change either substantially.
- **Do not decay biases and normalization parameters** (a widespread convention). They are exactly the 1-D parameters, so two parameter groups do the job. (So far, §4–5 passed `model.parameters()`, which decays everything with AdamW's default 0.01.)

In [ ]:
def param_groups(model, weight_decay):
    # Two groups: weight matrices/kernels are decayed; biases and norm parameters (1-D) are not.
    decay, no_decay = [], []
    for p in model.parameters():
        if p.requires_grad:
            (no_decay if p.ndim <= 1 else decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]

demo = nn.Sequential(nn.Linear(784, 256), nn.LayerNorm(256), nn.ReLU(), nn.Linear(256, 10))
opt = torch.optim.AdamW(param_groups(demo, weight_decay=0.05), lr=1e-3)
for i, g in enumerate(opt.param_groups):
    print(f"group {i}: {len(g['params'])} tensors, {sum(p.numel() for p in g['params']):,} values, "
          f"weight_decay={g['weight_decay']}")

---
# 7. Early stopping

## 7.1 The idea
The training loss keeps falling as long as you train. The **validation** loss usually falls, reaches a minimum, and then rises again as the model starts to memorize the training set (overfitting). Early stopping:

1. evaluates on the validation set after every epoch;
2. remembers the **best** weights seen so far;
3. stops when the monitored value has not improved by more than `min_delta` for `patience` epochs;
4. **restores the best weights** before the model is used.

It saves compute and acts as a regularizer, because it limits how far training moves the weights.

## 7.2 Doing it correctly: a checklist
- **Monitor a validation set, never the test set.** The test set is used once, at the very end.
- **Evaluate in evaluation mode:** `model.eval()` and `torch.no_grad()`, so dropout is off and BatchNorm uses its running statistics. Switch back with `model.train()` before training (our helpers do both).
- **Copy the best weights** (Bug 1 of §1.1). Keep the copy on the **CPU**, so that it does not double the model's GPU memory.
- **Restore the best weights** at the end; the last epoch is, by construction, `patience` epochs past the best one.
- **Prefer the validation loss** as the monitored value: it is smoother than accuracy and reacts earlier. Use `mode="max"` if you monitor accuracy.
- **Treat NaN as "no improvement"**, and fail with a clear message if no epoch ever improved (e.g. the run diverged from the start).
- **Choose `patience` larger than the epoch-to-epoch noise** and larger than the `patience` of a `ReduceLROnPlateau` scheduler, so the learning rate gets a chance to drop before training stops. Give both the **same definition of improvement**.
- **Pair it with a suitable schedule.** With a constant learning rate or `ReduceLROnPlateau`, early stopping works as intended. A cosine, WSD or OneCycle schedule is sized for the full run; stopping in the middle leaves it at a high learning rate. With those schedules, train to the end and keep the checkpoint with the best validation loss.
- **Save the best model to disk** as well, so a crash does not lose it.

In [ ]:
class EarlyStopping:
    # Stop when the monitored value has not improved by more than `min_delta` for `patience` epochs.
    # mode="min" for losses, "max" for accuracies. Keeps a CPU copy of the best weights and,
    # if `path` is given, also saves them to disk at every improvement.

    def __init__(self, patience=5, min_delta=0.0, mode="min", path=None):
        assert mode in ("min", "max")
        self.patience, self.min_delta, self.mode, self.path = patience, min_delta, mode, path
        self.best = math.inf if mode == "min" else -math.inf
        self.best_epoch, self.best_state, self.bad_epochs = None, None, 0

    def is_improvement(self, value):
        if not math.isfinite(value):                     # NaN or inf never counts as an improvement
            return False
        if self.mode == "min":
            return value < self.best - self.min_delta
        return value > self.best + self.min_delta

    def step(self, value, model, epoch):
        # Call once per epoch with the validation metric. Returns True when training should stop.
        if self.is_improvement(value):
            self.best, self.best_epoch, self.bad_epochs = value, epoch, 0
            # an independent COPY (not a reference), kept on the CPU
            self.best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
            if self.path is not None:
                torch.save({"epoch": epoch, "value": value, "model_state": self.best_state}, self.path)
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model):
        if self.best_state is None:
            raise RuntimeError("No epoch ever improved (e.g. every validation loss was NaN): nothing to restore.")
        model.load_state_dict(self.best_state)          # copies the values into the model's own tensors

A quick test with made-up validation losses (`patience=3`), and with a run whose losses are all NaN:

In [ ]:
stopper = EarlyStopping(patience=3)
dummy = nn.Linear(1, 1)
for epoch, val_loss in enumerate([1.0, 0.8, 0.7, 0.72, 0.69, 0.71, 0.70, 0.73, 0.75], start=1):
    stop = stopper.step(val_loss, dummy, epoch)
    print(f"epoch {epoch}: val loss {val_loss:.2f} | best {stopper.best:.2f} (epoch {stopper.best_epoch}) | "
          f"epochs without improvement: {stopper.bad_epochs}{'  -> STOP' if stop else ''}")
    if stop:
        break

diverged = EarlyStopping(patience=2)
for epoch in (1, 2):
    diverged.step(float("nan"), dummy, epoch)
try:
    diverged.restore_best(dummy)
except RuntimeError as e:
    print("all-NaN run:", e)

## 7.3 ⚠️ Why the copy matters

In [ ]:
net = nn.Linear(2, 1)
snapshot_wrong = net.state_dict()                   # references to the live parameters (views, §1.1)
snapshot_right = copy.deepcopy(net.state_dict())    # an independent copy

with torch.no_grad():
    net.weight += 1.0                               # "training continues" and changes the weights

print("wrong snapshot still equal to the CURRENT weights:", torch.equal(snapshot_wrong["weight"], net.weight))
print("right snapshot still equal to the CURRENT weights:", torch.equal(snapshot_right["weight"], net.weight))

The "wrong" snapshot silently followed the training: restoring it would give you the *last* weights, not the best ones.

## 7.4 Everything together on Fashion-MNIST
One real training run that uses the pieces of this notebook:
- **Model:** the MLP of §3.3 (He initialization for the hidden layers).
- **Optimizer:** **AdamW** with `lr=1e-3` from the sweep in §4.8, and **weight decay** 0.01 on the weight matrices only (§6).
- **Schedule:** **`ReduceLROnPlateau`** (§5) with `factor=0.5`, `patience=2`.
- **Early stopping** (§7) with `patience=6`, larger than the scheduler's, and at most 50 epochs.
- Scheduler and early stopping use the **same definition of improvement**: at least 0.001 lower validation loss (an absolute threshold).
- The best weights are also saved to `best_model.pt`.

In [ ]:
torch.manual_seed(0)
model = make_mlp()
loader = make_train_loader(seed=0)
WEIGHT_DECAY = 0.01
optimizer = torch.optim.AdamW(param_groups(model, WEIGHT_DECAY), lr=1e-3)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2,
                                           threshold=1e-3, threshold_mode="abs")
stopper = EarlyStopping(patience=6, min_delta=1e-3, path="best_model.pt")

history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}
decay_shrink = 1.0                                     # total shrink factor caused by weight decay alone
for epoch in range(1, 51):
    lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, loader, optimizer)
    decay_shrink *= (1 - lr * WEIGHT_DECAY) ** len(loader)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step(val_loss)                           # plateau scheduler: once per epoch, with the value
    for key, value in zip(history, (train_loss, val_loss, val_acc, lr)):
        history[key].append(value)
    improved = stopper.is_improvement(val_loss)
    print(f"epoch {epoch:2d} | lr {lr:.1e} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
          f"val acc {val_acc:.4f}{'  * best so far' if improved else ''}")
    if stopper.step(val_loss, model, epoch):
        print(f"Early stopping: no improvement for {stopper.patience} epochs.")
        break

stopper.restore_best(model)                            # keep the BEST epoch, not the last
print(f"Restored the weights of epoch {stopper.best_epoch} (val loss {stopper.best:.4f}).")

In [ ]:
epochs = range(1, len(history["val_loss"]) + 1)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 3.3))
a1.plot(epochs, history["train_loss"], "o-", ms=3, label="train")
a1.plot(epochs, history["val_loss"], "o-", ms=3, label="validation")
a2.plot(epochs, history["val_acc"], "o-", ms=3, color="tab:green")
a3.semilogy(epochs, history["lr"], "o-", ms=3, color="tab:purple")
for ax, title in zip((a1, a2, a3), ("cross-entropy loss", "validation accuracy", "learning rate (log)")):
    ax.axvline(stopper.best_epoch, color="red", ls="--", lw=1)
    ax.set_title(title); ax.set_xlabel("epoch")
a1.legend(); plt.tight_layout(); plt.show()

test_loss, test_acc = evaluate(model, test_loader)                        # the test set, used ONCE
print(f"Test accuracy of the restored model: {test_acc:.4f} (test loss {test_loss:.4f})")
print(f"Weight decay alone shrank the weights by a factor of {decay_shrink:.4f} over the whole run.")

**Reading the plots** (the red line marks the restored epoch). The training loss falls steadily, while the validation loss stalls after about three epochs: the model is starting to fit the training images more closely than it generalizes. In our run the plateau scheduler halved the learning rate after epoch 6, which produced the best validation loss at epoch 7. After that the validation loss did not improve again (it fluctuated around 0.38–0.41), early stopping ended the run at epoch 13, and the epoch-7 weights were restored (test accuracy about 86 %). Notice that validation *accuracy* still crept up slightly while the validation *loss* got worse: the model becomes over-confident on the examples it gets wrong. Loss and accuracy can disagree, and which one you monitor is a choice; the loss is the more sensitive signal of overfitting.

Two honest observations:
- **Weight decay played almost no role here.** The last line shows the total shrink it caused: well under 1 % over the whole run, because $\eta\lambda = 10^{-5}$ per step and the run is short. On this problem, early stopping is what controls overfitting. Weight decay matters more in long runs with many steps.
- **Early stopping is a safeguard, not a performance booster.** The fixed warm-up + cosine run of §5.5 reached a slightly lower validation loss in 10 epochs. Early stopping is most useful when you do not know how long to train.

## 7.5 Loading the best model from disk
`torch.load(..., weights_only=True)` (the default since PyTorch 2.6) only unpickles tensors and basic Python types, which is the safe way to load files:

In [ ]:
checkpoint = torch.load("best_model.pt", map_location=device, weights_only=True)
fresh = make_mlp()
fresh.load_state_dict(checkpoint["model_state"])
print(f"checkpoint from epoch {checkpoint['epoch']}: validation loss when saved {checkpoint['value']:.4f}, "
      f"after reloading {evaluate(fresh, val_loader)[0]:.4f}")

💡 To **resume** an interrupted run, rather than just keep the best model, a checkpoint needs everything that changes during training:
```python
torch.save({"epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),     # momentum / Adam buffers
            "scheduler_state": scheduler.state_dict(),
            "stopper": {"best": stopper.best, "best_epoch": stopper.best_epoch, "bad_epochs": stopper.bad_epochs},
            "rng_state": torch.get_rng_state()}, "last.pt")
```
Load each part back with the matching `load_state_dict` (and `torch.set_rng_state`) before continuing the loop.

**Takeaway.** Early stopping = validation loss + CPU copy of the best weights (also on disk) + patience larger than the scheduler's + restore at the end; the test set is used once.

---
# Summary

| Topic | Remember |
| --- | --- |
| Tensor internals | storage + offset + shape + strides; views alias memory; `view` needs a compatible layout, `reshape` copies if needed, `clone` always copies; no in-place writes into tensors autograd still needs. |
| `einsum` | shared letters multiply, letters missing after `->` are summed; `...` batches over leading dimensions. |
| Initialization | He (`kaiming_normal_`) for ReLU-type hidden layers, Xavier for tanh/sigmoid, default for the output layer; initial loss ≈ ln(classes). |
| Optimizers | AdamW (`lr=1e-3`, `weight_decay=0.01`) first; SGD + momentum 0.9 with a schedule for established vision recipes; compare optimizers only with tuned learning rates. |
| Learning rate | a short log-scale sweep on the validation set, then confirm with a full run. |
| Schedulers | cosine (+ warm-up) for a fixed budget, WSD for large pre-training, OneCycle for short runs (per batch, and it changes momentum), ReduceLROnPlateau for an unknown length; same unit for sizing and stepping; plot the lr. |
| Weight decay | `AdamW(weight_decay=λ)`, not `Adam(weight_decay=λ)`; skip biases and normalization parameters; total effect ∝ lr × λ × steps. |
| Early stopping | validation loss, `eval()` + `no_grad`, CPU copy of the best `state_dict` (and on disk), patience > scheduler patience, restore the best weights, test set once. |

## Further reading
- PyTorch documentation: [tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html), [`torch.einsum`](https://docs.pytorch.org/docs/stable/generated/torch.einsum.html), [`torch.nn.init`](https://docs.pytorch.org/docs/stable/nn.init.html), [`torch.optim` and learning-rate schedulers](https://docs.pytorch.org/docs/stable/optim.html), [saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).
- X. Glorot and Y. Bengio, *Understanding the difficulty of training deep feedforward neural networks*, AISTATS 2010.
- K. He et al., *Delving Deep into Rectifiers*, ICCV 2015 (He initialization).
- D. P. Kingma and J. Ba, *Adam: A Method for Stochastic Optimization*, ICLR 2015.
- I. Loshchilov and F. Hutter, *SGDR: Stochastic Gradient Descent with Warm Restarts*, ICLR 2017 (cosine schedule), and *Decoupled Weight Decay Regularization*, ICLR 2019 (AdamW).
- L. N. Smith and N. Topin, *Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates*, 2019 (one-cycle policy).
- S. Hu et al., *MiniCPM: Unveiling the Potential of Small Language Models with Scalable Training Strategies*, 2024 (WSD schedule).
- L. Prechelt, *Early Stopping — But When?*, in *Neural Networks: Tricks of the Trade*, 1998.